In [1]:
### Quick intros to python topics

In [1]:
from PySide6.QtWidgets import *
from PySide6.QtCore import *
from PySide6.QtGui import *
from BoardItem import *
from MyView import MyView
from BoardScene import BoardScene
from utils import Utils
from collections import defaultdict
import sys
from utils import CopperItemContainer ,LayerItem
from MainWindow import MainWindow

class ViaBase():#QGraphicsItem):
    def __init__(self, outerDiameter, innerDiameter, clearance, **kwargs):#,parent=None):
        print('VIABASE.KWARGS:', kwargs)
        super().__init__(**kwargs)#parent) 
        # super().__init__(parent=parent) TypeError: NO BAD no keywords use positional: LayersContainer.__init__() got an unexpected keyword argument 'parent' # IDK why this happens-- could not replicate in simple example. Something about QGraphicsItem preferring positional args. But sometimes it can take kwargs. I always put classes, which inherit QGI, LAST, in the inheritance, because super() cannot propagate correctly after it hits QGI.
        
        self._outerDiameter = outerDiameter 
        self._innerDiameter = innerDiameter 
        self._clearance = clearance

        self._boundingRect = QRectF(-(outerDiameter+clearance)/2 , -(outerDiameter+clearance)/2 , outerDiameter+clearance , outerDiameter+clearance) # Must include clearance in BR so we can redraw the clearance w/o artifacts.

    def boundingRect(self):
        return self._boundingRect 

    # def paint(self, painter, option, widget):
    #     pass 
    
    def shape(self): # Note that shape, .bR, are GOING to be reimplementing QGI.shape,.bR once ViaBase is inherited by Via, ViaItem. (Good, bad) practice? 
        path = QPainterPath()
        path.addEllipse(QPoint(),  self.outerRadius(), self.outerRadius() ) # This doesnt account for clearance, and it doesnt need to, but bR needs to
        return path 
        
    def outerDiameter(self):
        return self._outerDiameter 
    def innerDiameter(self):
        return self._innerDiameter 

    def outerRadius(self):
        return self._outerDiameter/2
    def innerRadius(self):
        return self._innerDiameter/2
    
    def clearance(self):
        return self._clearance
            
# class ViaItem(CopperItem, ViaBase):
class ViaItem(LayerItem, ViaBase, QGraphicsItem):
        # def QGI.__init__(self, parent: PySide6.QtWidgets.QGraphicsItem | None= ...) -> None: ...
    def __init__(self, layer, outerDiameter, innerDiameter, clearance, color , parent):
        # super().__init__(outerDiameter=outerDiameter, innerDiameter=innerDiameter, clearance=clearance, parent=parent)
        super().__init__( layer, outerDiameter=outerDiameter, innerDiameter=innerDiameter, clearance=clearance, parent=parent) # TypeError: ViaBase.__init__() takes 4 positional arguments but 5 were given
        # QGraphicsItem.__init__(self, parent)
        # print('VIAITEM.LAYER:', self.layer())
        self._color = color
        # self._boundingRect = QRectF(-(outerDiameter+clearance)/2 , -(outerDiameter+clearance)/2 , outerDiameter+clearance , outerDiameter+clearance) # Must include clearance in BR so we can redraw the clearance w/o artifacts.

    # def boundingRect(self): # Belive this covered by super
    #     return self._boundingRect
    
    def paint(self, painter, option, widget): # QGraphicsItem.paint reimplementation to draw the aspects of a via: a clearance indicator and some colored circles
        #DrawClearance
        painter.setPen(QPen(self._color, 0))
        painter.setBrush(Qt.BrushStyle.NoBrush)
        painter.drawEllipse(QPointF(), self.clearance()/2, self.clearance()/2)
        #DrawVia
        painter.setPen(Qt.NoPen)
        painter.setBrush(QBrush(self._color, bs=Qt.BrushStyle.SolidPattern))
        painter.drawEllipse(QPoint(0,0), self.outerDiameter()/2 , self.outerDiameter()/2)
        painter.setBrush(QColor(230,230,230)) # Gray
        painter.drawEllipse(QPoint(0,0), self.innerDiameter()/2+Utils.viaPlatingThickness , self.innerDiameter()/2+Utils.viaPlatingThickness)
        painter.setBrush(QColor(255,215,0)) # Gold
        painter.drawEllipse(QPoint(0,0), self.innerDiameter()/2, self.innerDiameter()/2)

# class Via(LayersContainer, ViaBase): # A Via is made up of several childItem viaItems, one viaItem per layer. 
class Via(ViaBase, CopperItemContainer, QGraphicsItem):
    
    def __init__(self,  outerDiameter, innerDiameter, clearance=Utils.viaClearance, layers=Utils.CopperLayers): # layers : A via may exist on all or some layers, default all # clearance: default 1mm
        print('VIA.MRO:', Via.mro())
        super().__init__(outerDiameter=outerDiameter, innerDiameter=innerDiameter, clearance=clearance, layers = layers)

        self.setFlags(QGraphicsItem.ItemIsMovable | QGraphicsItem.ItemIsSelectable)
        
        # print('VIA.LAYERS():', self.layers())
        for layer in self.layers():
            # print('LAYER:', layer)
            ViaItem(layer, outerDiameter, innerDiameter, clearance, Utils.layerColors[layer], self)
            # self.copperItems()[layer].append(item) Phasing out# Track ViaItem as copperItems ( is this necessary? )

    def nearestSceneSnap(self, pos): # Via only has one snap; center. But since TraceItem has two snaps, all items need this method to maintain consistent API.
        return self.scenePos()
    
    def mouseMoveEvent(self, event):
        super().mouseMoveEvent(event)
                
        self.setSceneTerminal()
        
    def net(self):
        return self._net 
    def setNet(self, net):
        # Note pad net is determined by the schematic connections. Via,Trace, net is determined by pads
        self._net = net
    def sceneTerminal(self): 
        return self.scenePos()
    def setSceneTerminal(self):
        self._sceneTerminal = self.scenePos()
        
    def sceneTerminals(self):
        return self._sceneTerminals
    def setSceneTerminals(self):
        self.setSceneTerminal()
        self._sceneTerminals = [self.sceneTerminal()]
    def boundingRect(self):
        return self.childrenBoundingRect() or QRectF()
    def paint(self, painter, option, widget):
        pass# ChildrenItems will paint themselves

from NetSymbol import NetSymbol
from Trace import Trace

window = MainWindow()
boardScene = window.centralWidget().widget(1).scene()
via = Via(50, 30, layers = ['F.Cu', 'B.Cu'])
trace = Trace(layers = ['F.Cu'], p1= QPointF(50,50), p2 =QPointF(1000, 500), traceWidth = 1)
boardScene.addItem(trace)

# print([item for item in via.childItems()])
ns = NetSymbol('?', 1, "symbols/NetSymbols/GND.sym")

via.setPos(100,100)
boardScene.addItem(via)
window.show()
sys.exit(qApp.exec())
# SELF.COMPONENTS: defaultdict(<class 'collections.defaultdict'>, {'?': defaultdict(None, {1: <Component.Component object at 0x00000272C7C6D790>}), 'GND': defaultdict(None, {})})


LAYERS: None
VIA.MRO: [<class '__main__.Via'>, <class '__main__.ViaBase'>, <class 'utils.CopperItemContainer'>, <class 'BoardItem.BoardItem'>, <class 'PySide6.QtWidgets.QGraphicsItem'>, <class 'Shiboken.Object'>, <class 'object'>]
VIABASE.KWARGS: {'layers': ['F.Cu', 'B.Cu']}
LAYERS: ['F.Cu', 'B.Cu']
VIABASE.KWARGS: {'parent': <__main__.Via(0x242d4f64c80, pos=0,0, flags=(ItemIsMovable|ItemIsSelectable)) at 0x00000242D508F600>}
VIABASE.KWARGS: {'parent': <__main__.Via(0x242d4f64c80, pos=0,0, flags=(ItemIsMovable|ItemIsSelectable)) at 0x00000242D508F600>}
INITIALIZED TRACEBASE 
P1: PySide6.QtCore.QPointF(50.000000, 50.000000)
P2: PySide6.QtCore.QPointF(1000.000000, 500.000000)
LAYERS: ['F.Cu']
SELF.LAYERS: ['F.Cu']
INITIALIZED TRACEBASE 
P1: PySide6.QtCore.QPointF(50.000000, 50.000000)
P2: PySide6.QtCore.QPointF(1000.000000, 500.000000)
LAYER: <class 'str'> F.Cu

DRAWBACKGROUND
XSCALE: 1.0
TICKSPACING: 17.86046511627907

DRAWBACKGROUND
XSCALE: 1.0
TICKSPACING: 17.86046511627907

DRAWBACKG

SystemExit: 0

c:\Users\robby\OneDrive\Saura\myenv\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# My Viewport dots are only appearing in one quadrant; xy positive. How do I get them in all quadrants? 
# A: offset by the exposed rect : painter.drawPoint( x*tickSpacing + rect.left()) , y*tickSpacing + rect.top()) 
# Great, but now there is a 'flowing' effect, appears as if dots flow behind items when zooming. (This because dots start drawing at rect leftTop. 
# A: Start drawing dots on grid, at grid pos nearest leftTop





In [ ]:
# My viewport is constrained to scroll only in a certain area. I want to be able to scroll on an effectively infinte area
# Answer: Set a large scroll area with :
#     view.setSceneRect(-10000, -10000, 20000, 20000)

# ### QGraphicsView ### 
# Visualize the contents of a QGraphicsScene in a scrollable viewport. 

# Scroll to any position on the scene using the scrollbars,(umm yet I cannot)(What is my sceen rectangle? unset, so infinite) or by calling .centerOn(QPoint)
# The visualized area is by default detected with QGS.itemsBoundingRect(), which returns the bounding rect of all items on the scene. 
# Use QGVIEW.setSceneRect() to set the visualized area yourself. This will adjust the scroll bars' ranges. 

# QGraphicsView.setBackground(painter, rect) default fills rect using the view's background brush. If no such brush defined(the default), the scene's .drawBackground is called instead. 

In [1]:
# Set Background to Draw Dots Sensibly
from utils import * 
from MyView import MyView
from SchematicScene import SchematicScene

class GridView(MyView ): 


    def drawBackground(self, painter, rect): 

        print()
        print('DRAWBACKGROUND')

        painter.setBrush(Qt.black)
        painter.setPen(QPen(Qt.black, 1)) # Note QPen width 1 makes dots much more visible than width 0 
        # painter.setPen(Qt.NoPen) Makes points disappear
        


        # Note WHen zooming wayin to wayout , the PEN WIDTH of your painted points becomes important so the user can see it. 
        # I'm content with this for prototype, but production app should dynamically set pen widths based on zoom (?)
        def calculateTickSpacing():
            xScale = painter.transform().m11() # xScale is represented at the transformationMatrix m11 element. 
            print('XSCALE:', xScale)
            tickSpacing = Utils.boardTickSpacing
            if (xScale <.5): # ZOOMEDWAYOUT
                painter.setPen(QPen(Qt.black, 10)) # Set a wide pen so user can still see the dots 
                tickSpacing = Utils.boardTickSpacing*10
            elif .5 <= xScale <= 10: 
                painter.setPen(QPen(Qt.black, 1))
                tickSpacing = Utils.boardTickSpacing
            if xScale > 20: #ZOOMEDWAYIN
                painter.setPen(QPen(Qt.black, .1)) # set a thin pen so user can still see the dots 
                tickSpacing = Utils.boardTickSpacing/10

            return tickSpacing

        tickSpacing = calculateTickSpacing()
        print('TICKSPACING:', tickSpacing)
        
        numTicksX = int(rect.width()/tickSpacing) 
        numTicksY = int(rect.height()/tickSpacing) 

        xStart = int(rect.left() / tickSpacing) * tickSpacing # important to start drawing points snapped to grid. If start drawing points at rect.left()&rect.top(), induces a 'flowing' effect while zooming
        yStart = int(rect.top() / tickSpacing) * tickSpacing
        
        for i in range(numTicksX): 
            for j in range(numTicksY):
                x = i*tickSpacing + xStart 
                y =  j* tickSpacing + yStart
                painter.drawPoint(QPointF(x , y)) # Note pass a QPointF() to be able to use floats with .drawPoint() 
                # painter.drawEllipse(QPointF(x,y), 1, 1)

    def wheelEvent(self, event): # Wheel as in mouseWheel 
        
        delta = event.angleDelta().y() # How much mouseWheel scrolled
        scaleFactor = math.pow(2.0, -delta / 500)
        self.scaleScene(scaleFactor)

    def scaleScene(self, scaleFactor):
        zoom = self.transform().scale(scaleFactor, scaleFactor).m11() # Scale current transform to predict zoom. The x scale lives in the matrix's m11 element      #  Used to do this , which also works: .mapRect(QRectF(0, 0, 1, 1)).width() # QTransform.mapRect(rect) -> QRectF, mapped onto the given QTransform. Note that we gave a unit rectangle; a rectangle where width&height=1. So, we are testing to see how much a unit scales under this transform. Note that self.transform() includes any previous scaling; representing the currently applied zoom, which we should limit to a certain range 

        if zoom < 0.01 or zoom > 100: # Prevent crazy scale changes.
            return

        self.scale(scaleFactor, scaleFactor) 



        
view = GridView() 
scene = SchematicScene()
view.setScene(scene)
r = QGraphicsRectItem(-100,-100, 200,200) 
r.setBrush(Qt.blue) 
r.setPen(Qt.NoPen)
scene.addItem(r)

view.show()

sys.exit(app.exec())







DRAWBACKGROUND
XSCALE: 1.0
TICKSPACING: 17.86046511627907

DRAWBACKGROUND
XSCALE: 1.0
TICKSPACING: 17.86046511627907

DRAWBACKGROUND
XSCALE: 1.0
TICKSPACING: 17.86046511627907

DRAWBACKGROUND
XSCALE: 1.0
TICKSPACING: 17.86046511627907

DRAWBACKGROUND
XSCALE: 1.0
TICKSPACING: 17.86046511627907

DRAWBACKGROUND
XSCALE: 1.0
TICKSPACING: 17.86046511627907

DRAWBACKGROUND
XSCALE: 1.0
TICKSPACING: 17.86046511627907

DRAWBACKGROUND
XSCALE: 1.0
TICKSPACING: 17.86046511627907

DRAWBACKGROUND
XSCALE: 1.0
TICKSPACING: 17.86046511627907

DRAWBACKGROUND
XSCALE: 1.0
TICKSPACING: 17.86046511627907

DRAWBACKGROUND
XSCALE: 1.0
TICKSPACING: 17.86046511627907

DRAWBACKGROUND
XSCALE: 1.0
TICKSPACING: 17.86046511627907

DRAWBACKGROUND
XSCALE: 1.0
TICKSPACING: 17.86046511627907

DRAWBACKGROUND
XSCALE: 1.0
TICKSPACING: 17.86046511627907

DRAWBACKGROUND
XSCALE: 1.0
TICKSPACING: 17.86046511627907

DRAWBACKGROUND
XSCALE: 1.0
TICKSPACING: 17.86046511627907

DRAWBACKGROUND
XSCALE: 1.0
TICKSPACING: 17.860465116279

SystemExit: 0

c:\Users\robby\OneDrive\Saura\myenv\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# Workflow porting NetSym from Kicad: 3V3 symbol
from utils import * 
from kicadSymbolConverter import KicadSymbolConverter
# from kicadFootprintConverter import KicadFootprintConverter

officialKicadSymbolsLibrariesPath = os.path.join('third_party', 'kicad', 'symbols', 'kicad-symbols')

threeV3FilePath = os.path.join(officialKicadSymbolsLibrariesPath , 'power.kicad_symdir', '+3V3.kicad_sym')
threeV3SymFile = KicadSymbolConverter.convert(threeV3FilePath, categories = ['netSymbols']) # places '+3V3.sym' into the 'netSymbols' folder.Note 'categories so named bc supposed to be like ['capacitors', 'ceramic_capacitors']




